In [1]:
import rasterio
import numpy as np
from pathlib import Path
from rasterio.warp import calculate_default_transform, reproject, Resampling

# Folders
input_folder = Path(r"D:/MyDrive/Stability/RawData/Monthly_Averages/MergedOutput_month_year")
coast_mask_file = Path(r"D:/MyDrive/Stability/Scripts/merged_coastlines.sdat")
output_folder = input_folder / "_GeoTIFFs_100m_gdal"
output_folder.mkdir(exist_ok=True)

# Target CRS (WGS84)
target_crs = "EPSG:4326"

# Read and reproject coast mask to WGS84
with rasterio.open(coast_mask_file) as mask_src:
    # Prepare transform and shape for the target CRS
    transform, width, height = calculate_default_transform(
        mask_src.crs, target_crs, mask_src.width, mask_src.height, *mask_src.bounds
    )
    profile = mask_src.profile.copy()
    profile.update({
        "crs": target_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    # Reproject
    mask_data_wgs84 = np.empty((height, width), dtype=mask_src.read(1).dtype)
    reproject(
        source=mask_src.read(1),
        destination=mask_data_wgs84,
        src_transform=mask_src.transform,
        src_crs=mask_src.crs,
        dst_transform=transform,
        dst_crs=target_crs,
        resampling=Resampling.nearest
    )

# Loop through all .tif files
for tif_file in input_folder.glob("*.tif"):
    with rasterio.open(tif_file) as src:
        raster_data = src.read(1)
        raster_profile = src.profile

        # Reproject mask to match raster if shape differs
        if (src.width, src.height) != mask_data_wgs84.shape[::-1]:
            mask_resampled = np.empty(raster_data.shape, dtype=mask_data_wgs84.dtype)
            reproject(
                mask_data_wgs84,
                mask_resampled,
                src_transform=transform,
                src_crs=target_crs,
                dst_transform=src.transform,
                dst_crs=src.crs,
                resampling=Resampling.nearest
            )
            mask_to_use = mask_resampled
        else:
            mask_to_use = mask_data_wgs84

        # Binary conversion: >0 & <1 -> 255, else 0
        binary = np.where((raster_data >= 0) & (raster_data < 1), 1, 0).astype(np.uint8)

        # Apply coast mask: keep only where mask == 0
        binary = np.where(mask_to_use == 0, binary, 0)

        # Update profile
        raster_profile.update(
            dtype=rasterio.uint8,
            count=1,
            nodata=None  # disable nodata handling
        )

        # Save output
        out_file = output_folder / tif_file.name
        with rasterio.open(out_file, "w", **raster_profile) as dst:
            dst.write(binary, 1)

    print(f"Processed {tif_file.name}")

Processed WeightedMean_2017_03.tif
Processed WeightedMean_2017_04.tif
Processed WeightedMean_2017_05.tif
Processed WeightedMean_2017_06.tif
Processed WeightedMean_2017_07.tif
Processed WeightedMean_2017_08.tif
Processed WeightedMean_2017_09.tif
Processed WeightedMean_2017_10.tif
Processed WeightedMean_2017_11.tif
Processed WeightedMean_2017_12.tif
Processed WeightedMean_2018_01.tif
Processed WeightedMean_2018_02.tif
Processed WeightedMean_2018_03.tif
Processed WeightedMean_2018_04.tif
Processed WeightedMean_2018_05.tif
Processed WeightedMean_2018_06.tif
Processed WeightedMean_2018_07.tif
Processed WeightedMean_2018_08.tif
Processed WeightedMean_2018_09.tif
Processed WeightedMean_2018_10.tif
Processed WeightedMean_2018_11.tif
Processed WeightedMean_2018_12.tif
Processed WeightedMean_2019_01.tif
Processed WeightedMean_2019_02.tif
Processed WeightedMean_2019_03.tif
Processed WeightedMean_2019_04.tif
Processed WeightedMean_2019_05.tif
Processed WeightedMean_2019_06.tif
Processed WeightedMe